# Face-expression observations directly from source videos

This notebook classifies **visible facial expressions** for face observations described only by metadata
(video + frame number + face bounding box). It never reads or writes face-crop image files.

```text
metadata row (video, frame_number, bbox, person/detection IDs, …)
      ↓  group by video → open each video once
      ↓  group by frame → decode each requested frame once (forward decoding across small gaps)
      ↓  crop every face in that frame: frame[y1:y2, x1:x2]  (BGR → RGB NumPy array)
      ↓  batch RGB arrays → AutoImageProcessor → AutoModelForImageClassification (GPU when available)
      ↓  raw expression scores + derived Positive / Negative / Neutral scores
      ↓  merged back onto the original metadata by row/detection ID → CSV
```

**Interpretation.** The model (`trpakov/vit-face-expression`) classifies the *appearance* of a face in a single frame
into expression categories (angry, disgust, fear, happy, neutral, sad, surprise). The `positive` / `negative` /
`neutral` "sentiment" columns are **derived visual-expression classifications** kept under those names for compatibility
with the rest of the hackathon pipeline. They do not measure, and must not be reported as, a person's actual internal
emotional state, mood, engagement, or intent. Treat every label as an unreviewed suggestion.

**Privacy.** Footage of young people stays in your authorized local/Colab environment. Face crops exist only in memory
(and in the inspection plots if you choose to display them); nothing is written to a `FaceCrops` folder. The model
weights are downloaded from Hugging Face on first use; images are never uploaded.

## 1 — Dependencies

Installs only packages that are missing, so an existing CUDA build of PyTorch (for example on a Colab GPU runtime)
is never replaced.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    # import name: pip package
    "transformers": "transformers",
    "torch": "torch",
    "cv2": "opencv-python-headless",
    "pandas": "pandas",
    "tqdm": "tqdm",
    "matplotlib": "matplotlib",
}

missing = [pkg for module, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")
# Reading .xlsx metadata additionally needs `openpyxl`.

## 1b — Colab setup (skipped automatically outside Colab)

A Colab runtime starts empty and is wiped when it disconnects. On Colab this cell:

1. clones the repository (public) to get `data/analysis.sqlite` and the repo layout,
2. mounts Google Drive and copies the videos listed in the database's `Video` table from `COLAB_DRIVE_VIDEO_DIR`
   into `data/videos/` on the runtime's local disk (faster to decode than reading from Drive), skipping files already
   copied,
3. changes the working directory to the clone so the configuration cell finds it automatically.

Set `COLAB_DRIVE_VIDEO_DIR` to the Drive folder holding the original `.mp4` files (exact filenames as in the database).
If `drive.mount()` doesn't work from VS Code, mount Drive once from colab.research.google.com, or put the videos into
`/content/ai_for_good_2026/data/videos/` another way; missing videos are logged as errors, not fatal.

In [ ]:
import importlib.util
import os
import shutil
import sqlite3
import subprocess
from contextlib import closing
from pathlib import Path

IN_COLAB = Path("/content").is_dir() and importlib.util.find_spec("google.colab") is not None

COLAB_REPO_URL = "https://github.com/TheAdaptoid/ai_for_good_2026.git"
COLAB_REPO_BRANCH = "caden/identity-pipeline"   # switch to "main" once PR #1 is merged
COLAB_REPO_DIR = Path("/content/ai_for_good_2026")
COLAB_DRIVE_VIDEO_DIR = Path("/content/drive/MyDrive/Hackathon_Videos")
COLAB_DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/sentiment_outputs")  # results backup; None to skip
COPY_VIDEOS_TO_LOCAL_DISK = True                # False = symlink to Drive (no copy, slower decoding)

if not IN_COLAB:
    print("Not running on Colab: setup skipped.")
else:
    database = COLAB_REPO_DIR / "data" / "analysis.sqlite"
    if not database.is_file():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", COLAB_REPO_BRANCH,
                        COLAB_REPO_URL, str(COLAB_REPO_DIR)], check=True)
    print("Database:", database, f"({database.stat().st_size / 1e6:.1f} MB)")

    with closing(sqlite3.connect(database.as_uri() + "?mode=ro", uri=True)) as con:
        wanted = [name for (name,) in con.execute("SELECT filename FROM Video ORDER BY video_id")]

    video_dir = COLAB_REPO_DIR / "data" / "videos"
    video_dir.mkdir(parents=True, exist_ok=True)
    if not Path("/content/drive/MyDrive").is_dir():
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as exc:
            print(f"Could not mount Google Drive ({exc!r}). Put the videos in {video_dir} another way.")

    if "<" in str(COLAB_DRIVE_VIDEO_DIR):
        print("Set COLAB_DRIVE_VIDEO_DIR to the Drive folder with the videos, then rerun this cell.")
    elif not COLAB_DRIVE_VIDEO_DIR.is_dir():
        print("Drive video folder not found:", COLAB_DRIVE_VIDEO_DIR)
    else:
        for name in wanted:
            source, target = COLAB_DRIVE_VIDEO_DIR / name, video_dir / name
            if not source.is_file():
                print("  MISSING in Drive:", name)
            elif target.exists() and target.stat().st_size == source.stat().st_size:
                print("  already present:", name)
            elif COPY_VIDEOS_TO_LOCAL_DISK:
                print(f"  copying {name} ({source.stat().st_size / 1e9:.2f} GB)…", flush=True)
                shutil.copyfile(source, target)
            else:
                target.unlink(missing_ok=True)
                target.symlink_to(source)
                print("  linked:", name)

    present = [name for name in wanted if (video_dir / name).exists()]
    print(f"Videos available: {len(present)} / {len(wanted)}")
    os.chdir(COLAB_REPO_DIR)
    print("Working directory:", Path.cwd())

## 2 — Configuration

Every path derives from `PROJECT_DIR`, which is found automatically as the nearest folder (the current one or a parent)
containing `data/analysis.sqlite`, so it works from `notebooks/` or from the repo root. On a Colab runtime the
repo layout must exist **on that runtime** (uploaded folder or mounted Google Drive):

```text
<PROJECT_DIR>/
├── data/analysis.sqlite     # schema database: Video, Frame, Person, FaceDetection
└── data/videos/*.mp4        # original videos (Video.file_path = data/videos/<filename>)
```

By default the observations come from the schema: `FaceDetection ⋈ Frame ⋈ Video`, one row per `detection_id`
(the key the `Engagement` table joins on), with `person_id`, `frame_number`, `source_time_seconds`, the bbox, and
`det_score`. A CSV/XLSX with video, frame, and bbox columns still works as an alternative `METADATA_FILE`.

In [ ]:
from pathlib import Path

SCHEMA_DB = Path("data") / "analysis.sqlite"


def find_project_dir():
    """The nearest folder (current or parent) containing data/analysis.sqlite."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / SCHEMA_DB).is_file():
            return folder
    raise FileNotFoundError(
        f"No {SCHEMA_DB} found in {Path.cwd()} or its parents. Set PROJECT_DIR explicitly below.")


PROJECT_DIR = find_project_dir()
# PROJECT_DIR = Path("/content/ai_for_good_2026")                # Colab runtime, uploaded/cloned repo
# PROJECT_DIR = Path("/content/drive/MyDrive/ai_for_good_2026")  # Colab runtime, mounted Google Drive

# Schema database (FaceDetection ⋈ Frame ⋈ Video); videos resolve from Video.file_path (data/videos/...).
METADATA_FILE = PROJECT_DIR / SCHEMA_DB
# METADATA_FILE = PROJECT_DIR / "sentiment_metadata.csv"       # alternative: CSV/XLSX with video, frame, bbox columns

# Searched (in order) for video filenames that don't resolve directly; recursive search is a last resort.
VIDEO_SEARCH_DIRS = [
    PROJECT_DIR / "data" / "videos",
    PROJECT_DIR / "videos",
    PROJECT_DIR / "data",
    PROJECT_DIR,
]
VIDEO_EXTENSIONS = {".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm", ".mpg", ".mpeg"}

MODEL_NAME = "trpakov/vit-face-expression"

# 0 = metadata frame numbers are 0-based source-frame indices. Use -1 if they start at 1.
FRAME_INDEX_OFFSET = 0
# 0.0 = exact face boxes; 0.1 adds 10% of the box size on each side.
BBOX_PADDING_FRACTION = 0.0

GPU_BATCH_SIZE = 64
CPU_BATCH_SIZE = 16
USE_FP16_ON_CUDA = True

# Requested frames at most this far ahead are reached by decoding forward (exact and cheap for nearby frames);
# larger jumps seek instead.
MAX_SEQUENTIAL_GAP = 300

MAX_OBSERVATIONS = None          # e.g. 500 while testing; None = every observation
BENCHMARK_OBSERVATIONS = 200
TEST_OBSERVATION_INDEX = 0       # row position used by the single-observation test cell
TOP_N = 5                        # faces per inspection group

OUTPUT_DIR = PROJECT_DIR / "sentiment_outputs"
FINAL_CSV = OUTPUT_DIR / "face_expression_observations_from_videos.csv"
ERROR_CSV = OUTPUT_DIR / "face_expression_errors.csv"
SAMPLE_CSV = OUTPUT_DIR / "face_expression_observations_sample.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_DIR:  ", PROJECT_DIR.resolve())
print("METADATA_FILE:", METADATA_FILE)
print("OUTPUT_DIR:   ", OUTPUT_DIR.resolve())

## 3 — Load metadata and detect columns

Recognized column names (case-insensitive):

| Purpose | Names |
|---|---|
| Video | `video_path`, `source_video_path`, `video_filename`, `video_file`, `source_video`, `video` |
| Frame | `frame_number`, `frame_index`, `frame_idx`, `source_frame_index`, `frame` |
| Box (4 columns) | `x1/y1/x2/y2`, `face_bbox_x1…`, `bbox_x1…`, `xmin/ymin/xmax/ymax`, `left/top/right/bottom` |
| Box (1 column) | `bbox`, `face_bbox`, `bounding_box`, `box`, `xyxy` containing `[x1, y1, x2, y2]` |
| Stable ID | `detection_id`, `observation_id` (otherwise the row position) |

Pixel and normalized (`0–1`) coordinates are both supported.

In [ ]:
import ast
import re
import sqlite3
from contextlib import closing

import numpy as np
import pandas as pd
from IPython.display import display

SQLITE_QUERY = """
SELECT d.detection_id, d.person_id, d.frame_id, f.video_id,
       v.title AS video_title, v.filename AS video_filename, v.file_path AS video_path,
       f.frame_number, f.source_time_seconds,
       d.x1, d.y1, d.x2, d.y2, d.det_score
FROM FaceDetection AS d
JOIN Frame AS f ON f.frame_id = d.frame_id
JOIN Video AS v ON v.video_id = f.video_id
ORDER BY f.video_id, f.frame_number, d.detection_id
"""

if not METADATA_FILE.exists():
    raise FileNotFoundError(f"Metadata file not found: {METADATA_FILE}\nSet METADATA_FILE in the configuration cell.")

suffix = METADATA_FILE.suffix.lower()
if suffix == ".csv":
    metadata_df = pd.read_csv(METADATA_FILE)
elif suffix in {".xlsx", ".xls"}:
    metadata_df = pd.read_excel(METADATA_FILE)
elif suffix in {".sqlite", ".sqlite3", ".db"}:
    with closing(sqlite3.connect(METADATA_FILE.resolve().as_uri() + "?mode=ro", uri=True)) as con:
        metadata_df = pd.read_sql_query(SQLITE_QUERY, con)
else:
    raise ValueError("METADATA_FILE must be CSV, XLSX/XLS, or a SQLite database (.sqlite/.db).")

metadata_df = metadata_df.reset_index(drop=True)
metadata_df["_observation_row"] = np.arange(len(metadata_df), dtype=np.int64)


def find_column(df, candidates):
    lookup = {str(c).strip().casefold(): c for c in df.columns}
    for name in candidates:
        if name.casefold() in lookup:
            return lookup[name.casefold()]
    return None


VIDEO_COL = find_column(metadata_df, ["video_path", "source_video_path", "video_filename", "video_file",
                                      "source_video", "video"])
FRAME_COL = find_column(metadata_df, ["frame_number", "frame_index", "frame_idx", "source_frame_index", "frame"])
X1_COL = find_column(metadata_df, ["x1", "face_bbox_x1", "bbox_x1", "face_x1", "xmin", "x_min", "left"])
Y1_COL = find_column(metadata_df, ["y1", "face_bbox_y1", "bbox_y1", "face_y1", "ymin", "y_min", "top"])
X2_COL = find_column(metadata_df, ["x2", "face_bbox_x2", "bbox_x2", "face_x2", "xmax", "x_max", "right"])
Y2_COL = find_column(metadata_df, ["y2", "face_bbox_y2", "bbox_y2", "face_y2", "ymax", "y_max", "bottom"])
BBOX_COL = find_column(metadata_df, ["bbox", "face_bbox", "bounding_box", "box", "xyxy"])
OBS_ID_COL = find_column(metadata_df, ["detection_id", "observation_id"])
HAS_FOUR_BBOX_COLS = all(c is not None for c in (X1_COL, Y1_COL, X2_COL, Y2_COL))

if VIDEO_COL is None:
    raise ValueError(f"Could not identify the video column. Columns: {metadata_df.columns.tolist()}")
if FRAME_COL is None:
    raise ValueError(f"Could not identify the frame-number column. Columns: {metadata_df.columns.tolist()}")
if not HAS_FOUR_BBOX_COLS and BBOX_COL is None:
    raise ValueError("Could not find bounding-box columns (x1/y1/x2/y2 or a combined bbox column). "
                     f"Columns: {metadata_df.columns.tolist()}")

print("Observations:", f"{len(metadata_df):,}")
print("Video column:", VIDEO_COL)
print("Frame column:", FRAME_COL)
print("BBox:", (X1_COL, Y1_COL, X2_COL, Y2_COL) if HAS_FOUR_BBOX_COLS else BBOX_COL)
print("Stable observation ID:", OBS_ID_COL or "row position (_observation_row)")
display(metadata_df.head(3))

## 4 — Locate videos, select the device, and load the model

The model is loaded with `AutoImageProcessor` + `AutoModelForImageClassification`, so batches of RGB NumPy arrays go
straight into the processor. Device priority: CUDA → Apple MPS → CPU. On CUDA, inference runs under
`torch.inference_mode()` with FP16 autocast (softmax is computed in FP32).

Videos that can't be found are logged per observation in the error table instead of stopping the run.

In [ ]:
import cv2
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification

_recursive_video_index = None


def _video_index():
    global _recursive_video_index
    if _recursive_video_index is None:
        _recursive_video_index = {}
        for root in VIDEO_SEARCH_DIRS:
            if root.exists():
                for p in root.rglob("*"):
                    if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS:
                        _recursive_video_index.setdefault(p.name.casefold(), set()).add(p.resolve())
    return _recursive_video_index


def resolve_video_path(value):
    value = str(value).replace("\\", "/").strip()
    direct = [Path(value), PROJECT_DIR / value] + [root / Path(value).name for root in VIDEO_SEARCH_DIRS]
    for candidate in direct:
        if candidate.is_file():
            return candidate.resolve()
    matches = sorted(_video_index().get(Path(value).name.casefold(), ()))
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError(f"Several videos are named {Path(value).name!r}: {matches}. Use a more specific path.")
    raise FileNotFoundError(f"Could not locate video {value!r} under {[str(p) for p in VIDEO_SEARCH_DIRS]}")


resolved_video_paths = {}
video_resolution_errors = {}
for value in metadata_df[VIDEO_COL].dropna().astype(str).unique():
    try:
        resolved_video_paths[value] = resolve_video_path(value)
    except (FileNotFoundError, ValueError) as exc:
        video_resolution_errors[value] = str(exc)

print(f"Videos resolved: {len(resolved_video_paths)}  |  missing/ambiguous: {len(video_resolution_errors)}")
for value, path in list(resolved_video_paths.items())[:12]:
    print("  •", value, "->", path)
for value, message in video_resolution_errors.items():
    print("  ✗", message)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

BATCH_SIZE = GPU_BATCH_SIZE if DEVICE.type in {"cuda", "mps"} else CPU_BATCH_SIZE
USE_AMP = DEVICE.type == "cuda" and USE_FP16_ON_CUDA

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageClassification.from_pretrained(MODEL_NAME).to(DEVICE).eval()
ID2LABEL = {int(k): str(v) for k, v in model.config.id2label.items()}

print("\nModel:", MODEL_NAME)
print("Device:", DEVICE, f"({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else "")
print("Batch size:", BATCH_SIZE, "| FP16 autocast:", USE_AMP)
print("Labels:", ID2LABEL)

## 5 — Frame reading, cropping, inference, and scoring helpers

- `FrameReader` keeps one `cv2.VideoCapture` open per video. Requested frames are visited in ascending order; nearby
  frames are reached by decoding forward with `grab()` (no colour conversion), larger jumps by seeking.
- `crop_face_rgb()` returns an RGB `uint8` NumPy array shaped `(height, width, 3)`.
- `prediction_scores()` is unchanged from the original notebook:
  - Positive-expression score = `happy`
  - Negative-expression score = `sad + angry + fear + disgust`
  - Neutral-expression score = `neutral`
  - The three are normalized to sum to 1; `surprise` is kept separately. Surprise ≥ 0.30 or a top-two margin < 0.15
    marks the row `Uncertain` / `Review carefully`.

In [ ]:
import math


class FrameReader:
    """One open video; read(index) returns the BGR frame at a 0-based index."""

    def __init__(self, path):
        self.path = Path(path)
        self.cap = cv2.VideoCapture(str(self.path))
        if not self.cap.isOpened():
            raise RuntimeError(f"Could not open video (missing or corrupted): {self.path}")
        self.next_index = 0  # index the next cap.read() returns; None = unknown, seek next time

    def read(self, index):
        index = int(index)
        gap = None if self.next_index is None else index - self.next_index
        if gap is not None and 0 <= gap <= MAX_SEQUENTIAL_GAP:
            for _ in range(gap):
                if not self.cap.grab():
                    self.next_index = None
                    raise RuntimeError(f"Video ended before frame {index}")
        else:
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, index)
        ok, frame = self.cap.read()
        if not ok or frame is None:
            self.next_index = None
            raise RuntimeError(f"Could not read frame {index}")
        self.next_index = index + 1
        return frame

    def close(self):
        self.cap.release()

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        self.close()


def clean_label(label):
    return re.sub(r"[^a-z]", "", str(label).lower())


def parse_bbox_value(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        values = list(value)
    else:
        text = str(value).strip()
        try:
            parsed = ast.literal_eval(text)
            values = list(parsed) if isinstance(parsed, (list, tuple)) else []
        except (ValueError, SyntaxError):
            values = re.findall(r"[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?", text)
    if len(values) != 4:
        raise ValueError(f"Expected 4 bbox values, got {value!r}")
    return [float(v) for v in values]


def bbox_from_row(row):
    if HAS_FOUR_BBOX_COLS:
        return [float(row[X1_COL]), float(row[Y1_COL]), float(row[X2_COL]), float(row[Y2_COL])]
    return parse_bbox_value(row[BBOX_COL])


def normalize_and_clip_bbox(bbox, frame_width, frame_height, padding_fraction=0.0):
    x1, y1, x2, y2 = [float(v) for v in bbox]
    if not all(math.isfinite(v) for v in (x1, y1, x2, y2)):
        raise ValueError(f"Non-finite bbox {bbox}")
    if x2 <= x1 or y2 <= y1:
        raise ValueError(f"Invalid bbox (needs x1 < x2 and y1 < y2): {bbox}")
    if 0.0 <= min(x1, y1, x2, y2) and max(x1, y1, x2, y2) <= 1.0:  # normalized coordinates
        x1, x2 = x1 * frame_width, x2 * frame_width
        y1, y2 = y1 * frame_height, y2 * frame_height
    pad_x, pad_y = (x2 - x1) * padding_fraction, (y2 - y1) * padding_fraction
    # Round outward, then clip to the frame.
    left = max(0, min(frame_width, math.floor(x1 - pad_x)))
    top = max(0, min(frame_height, math.floor(y1 - pad_y)))
    right = max(0, min(frame_width, math.ceil(x2 + pad_x)))
    bottom = max(0, min(frame_height, math.ceil(y2 + pad_y)))
    if right <= left or bottom <= top:
        raise ValueError(f"Empty crop after clipping {bbox} to {frame_width}x{frame_height}")
    return left, top, right, bottom


def crop_face_rgb(frame_bgr, bbox):
    height, width = frame_bgr.shape[:2]
    x1, y1, x2, y2 = normalize_and_clip_bbox(bbox, width, height, BBOX_PADDING_FRACTION)
    face_bgr = frame_bgr[y1:y2, x1:x2]
    if face_bgr.size == 0:
        raise ValueError("Empty face crop")
    return cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB), (x1, y1, x2, y2)


def infer_numpy_batch(face_rgb_arrays):
    """List of HxWx3 RGB uint8 arrays -> list of [{'label', 'score'}, ...] per face."""
    if not face_rgb_arrays:
        return []
    inputs = processor(images=face_rgb_arrays, return_tensors="pt", input_data_format="channels_last")
    pixel_values = inputs["pixel_values"].to(DEVICE)
    with torch.inference_mode():
        if USE_AMP:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(pixel_values=pixel_values).logits
        else:
            logits = model(pixel_values=pixel_values).logits
    probabilities = torch.softmax(logits.float(), dim=-1).cpu().numpy()
    return [[{"label": ID2LABEL[i], "score": float(p[i])} for i in range(len(p))] for p in probabilities]


def prediction_scores(predictions):
    raw = {clean_label(p["label"]): float(p["score"]) for p in predictions}
    aliases = {
        "angry": ("angry", "anger"),
        "disgust": ("disgust", "disgusted"),
        "fear": ("fear", "fearful"),
        "happy": ("happy", "happiness"),
        "neutral": ("neutral",),
        "sad": ("sad", "sadness"),
        "surprise": ("surprise", "surprised"),
    }

    def get_score(group):
        for candidate in aliases[group]:
            if candidate in raw:
                return raw[candidate]
        return 0.0

    happy = get_score("happy")
    negative = sum(get_score(k) for k in ("sad", "angry", "fear", "disgust"))
    neutral = get_score("neutral")
    surprise = get_score("surprise")
    total = happy + negative + neutral

    if total > 0:
        positive_score, negative_score, neutral_score = happy / total, negative / total, neutral / total
    else:
        positive_score = negative_score = neutral_score = float("nan")

    grouped = {"Positive": positive_score, "Negative": negative_score, "Neutral": neutral_score}
    ordered = sorted(grouped.items(), key=lambda x: x[1], reverse=True)
    top_label, top_score = ordered[0]
    margin = ordered[0][1] - ordered[1][1]
    ambiguous = surprise >= 0.30 or margin < 0.15

    return {
        "raw_happy_score": happy,
        "raw_sad_score": get_score("sad"),
        "raw_angry_score": get_score("angry"),
        "raw_fear_score": get_score("fear"),
        "raw_disgust_score": get_score("disgust"),
        "raw_neutral_score": neutral,
        "raw_surprise_score": surprise,
        "positive_score": positive_score,
        "negative_score": negative_score,
        "neutral_score": neutral_score,
        "suggested_label": "Uncertain" if ambiguous else top_label,
        "top_derived_score": top_score,
        "top_vs_second_margin": margin,
        "review_flag": "Review carefully" if ambiguous else "Suggested label",
    }


def read_face_for_row(row):
    """Reconstruct one face (RGB array) from video + frame + bbox; used by the test and inspection cells."""
    video_value = str(row[VIDEO_COL])
    if video_value not in resolved_video_paths:
        raise FileNotFoundError(video_resolution_errors.get(video_value, f"Unresolved video {video_value!r}"))
    video_path = resolved_video_paths[video_value]
    frame_index = int(row[FRAME_COL]) + FRAME_INDEX_OFFSET
    bbox = bbox_from_row(row)
    with FrameReader(video_path) as reader:
        frame_bgr = reader.read(frame_index)
    face_rgb, clipped_bbox = crop_face_rgb(frame_bgr, bbox)
    return {"video_path": video_path, "frame_index": frame_index, "bbox": bbox,
            "clipped_bbox": clipped_bbox, "face_rgb": face_rgb}

## 6 — Test one observation

Check that the metadata points at the intended person before processing everything. Scores are model outputs, not
calibrated probabilities of an actual emotion.

In [ ]:
import matplotlib.pyplot as plt

test_row = metadata_df.iloc[TEST_OBSERVATION_INDEX]
test = read_face_for_row(test_row)
test_predictions = infer_numpy_batch([test["face_rgb"]])[0]
test_derived = prediction_scores(test_predictions)

print("Observation ID: ", test_row[OBS_ID_COL] if OBS_ID_COL else int(test_row["_observation_row"]))
print("Video path:     ", test["video_path"])
print("Frame number:   ", test["frame_index"])
print("Original bbox:  ", test["bbox"])
print("Clipped bbox:   ", test["clipped_bbox"])
print("NumPy crop shape:", test["face_rgb"].shape)
print("dtype:          ", test["face_rgb"].dtype)

plt.figure(figsize=(3, 3))
plt.imshow(test["face_rgb"])
plt.axis("off")
plt.title(f"{test_derived['suggested_label']}", fontsize=10)
plt.show()

print("\nRaw facial-expression probabilities:")
for p in sorted(test_predictions, key=lambda x: x["score"], reverse=True):
    print(f"  {p['label']:10s} {p['score']:.4f}")

print("\nDerived visual-expression scores:")
for key in ("positive_score", "negative_score", "neutral_score", "raw_surprise_score",
            "top_vs_second_margin", "suggested_label", "review_flag"):
    value = test_derived[key]
    print(f"  {key:22s} {value:.4f}" if isinstance(value, float) else f"  {key:22s} {value}")

## 7 — Batched processing function

- observations are grouped by video (each video opened once) and by frame (each frame decoded once; every face in that
  frame is cropped from the same matrix)
- crops accumulate into batches of `BATCH_SIZE` for inference
- any per-observation, per-frame, per-video, or per-batch failure is logged in the error table and the run continues

In [ ]:
from tqdm.auto import tqdm

ERROR_COLUMNS = ["observation_row", "observation_id", "video", "frame_number", "error"]


def run_observations(source_df, show_progress=True):
    rows, errors = [], []
    pending_faces, pending_info = [], []

    def add_error(record, video, frame_number, message):
        errors.append({
            "observation_row": int(record["_observation_row"]),
            "observation_id": record[OBS_ID_COL] if OBS_ID_COL else int(record["_observation_row"]),
            "video": str(video),
            "frame_number": frame_number,
            "error": message,
        })

    def flush_batch():
        if not pending_faces:
            return
        try:
            batch_predictions = infer_numpy_batch(pending_faces)
        except Exception as exc:  # e.g. CUDA out of memory: log the batch, keep going
            for info in pending_info:
                add_error(info["record"], info["video"], info["frame_number"], f"Inference failed: {exc!r}")
        else:
            for info, predictions in zip(pending_info, batch_predictions):
                x1, y1, x2, y2 = info["bbox_used"]
                rows.append({
                    "_observation_row": int(info["record"]["_observation_row"]),
                    "model_name": MODEL_NAME,
                    "bbox_x1_used": x1, "bbox_y1_used": y1, "bbox_x2_used": x2, "bbox_y2_used": y2,
                    **prediction_scores(predictions),
                    **{f"raw_model_{clean_label(p['label'])}_score": p["score"] for p in predictions},
                    "review_status": "Pending",
                    "reviewer_label": "",
                    "reviewer_notes": "",
                })
        pending_faces.clear()
        pending_info.clear()

    work = source_df.copy()
    work["_video_path"] = work[VIDEO_COL].astype(str).map(resolved_video_paths)
    frames = pd.to_numeric(work[FRAME_COL], errors="coerce") + FRAME_INDEX_OFFSET
    valid_frame = frames.notna() & (frames >= 0) & (frames == frames.round())
    work["_frame_index"] = frames.where(valid_frame, -1).astype(np.int64)

    progress = tqdm(total=len(work), desc="Observations", disable=not show_progress)

    for record in work[work["_video_path"].isna()].to_dict("records"):
        video_value = str(record[VIDEO_COL])
        add_error(record, video_value, record[FRAME_COL],
                  video_resolution_errors.get(video_value, "Missing video"))
        progress.update(1)
    for record in work[work["_video_path"].notna() & ~valid_frame].to_dict("records"):
        add_error(record, record["_video_path"], record[FRAME_COL], "Invalid frame number")
        progress.update(1)

    valid = work[work["_video_path"].notna() & valid_frame]
    for video_path, video_df in valid.groupby("_video_path", sort=False):
        try:
            reader = FrameReader(video_path)
        except RuntimeError as exc:
            for record in video_df.to_dict("records"):
                add_error(record, video_path, int(record["_frame_index"]), str(exc))
            progress.update(len(video_df))
            continue

        with reader:
            for frame_index, frame_df in video_df.groupby("_frame_index", sort=True):
                records = frame_df.to_dict("records")
                try:
                    frame_bgr = reader.read(frame_index)
                except Exception as exc:
                    for record in records:
                        add_error(record, video_path, int(frame_index), str(exc))
                    progress.update(len(records))
                    continue

                for record in records:  # every face in this frame reuses frame_bgr
                    try:
                        face_rgb, bbox_used = crop_face_rgb(frame_bgr, bbox_from_row(record))
                    except Exception as exc:
                        add_error(record, video_path, int(frame_index), str(exc))
                        continue
                    pending_faces.append(face_rgb)
                    pending_info.append({"record": record, "video": video_path,
                                         "frame_number": int(frame_index), "bbox_used": bbox_used})
                    if len(pending_faces) >= BATCH_SIZE:
                        flush_batch()
                progress.update(len(records))

    flush_batch()
    progress.close()
    return pd.DataFrame(rows), pd.DataFrame(errors, columns=ERROR_COLUMNS)

## 8 — Benchmark and estimate the full runtime

Times the complete pipeline (video access, frame decoding, NumPy cropping, preprocessing, inference, result creation)
on the first `BENCHMARK_OBSERVATIONS` real observations, after one warm-up batch so model initialization and CUDA
kernel selection don't distort the estimate.

Decoding cost depends on how densely the requested frames are spread through each video, so treat the estimate as a
guide: allow extra time for long videos with sparse observations, storage speed (Drive is slower than local disk),
and first-time model download.

In [ ]:
import time

work_metadata_df = metadata_df if MAX_OBSERVATIONS is None else metadata_df.head(int(MAX_OBSERVATIONS))
work_metadata_df = work_metadata_df.copy()
benchmark_df = work_metadata_df.head(BENCHMARK_OBSERVATIONS).copy()

# Warm-up with one real face, repeated to a full batch.
for _, warm_row in benchmark_df.iterrows():
    try:
        warm_face = read_face_for_row(warm_row)["face_rgb"]
    except Exception:
        continue
    infer_numpy_batch([warm_face] * BATCH_SIZE)
    break

if DEVICE.type == "cuda":
    torch.cuda.synchronize()
start = time.perf_counter()
bench_results, bench_errors = run_observations(benchmark_df, show_progress=False)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
benchmark_seconds = time.perf_counter() - start

observations_processed = len(bench_results) + len(bench_errors)
rate = observations_processed / benchmark_seconds if benchmark_seconds > 0 else float("nan")
total_observations = len(work_metadata_df)
estimated_seconds = total_observations / rate if rate > 0 else float("nan")

print(f"Benchmark observations: {observations_processed} ({len(bench_results)} scored, {len(bench_errors)} errors)")
print(f"Benchmark time: {benchmark_seconds:.2f} seconds")
print(f"Observations per second: {rate:.2f}")
print(f"Total observations: {total_observations:,}")
print(f"Estimated full runtime in minutes: {estimated_seconds / 60:.1f}")
print(f"Estimated full runtime in hours: {estimated_seconds / 3600:.2f}")

## 9 — Run the complete dataset

In [ ]:
start = time.perf_counter()
results_df, errors_df = run_observations(work_metadata_df, show_progress=True)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
elapsed = time.perf_counter() - start

print(f"Scored observations: {len(results_df):,} / {len(work_metadata_df):,}")
print(f"Errors: {len(errors_df):,}")
print(f"Elapsed: {elapsed / 60:.2f} minutes ({(len(results_df) + len(errors_df)) / max(elapsed, 1e-9):.2f} observations/second)")
display(results_df.head(3))
if not errors_df.empty:
    display(errors_df["error"].value_counts().head(10).to_frame("count"))
    display(errors_df.head(10))

## 10 — Merge predictions onto the original metadata and save

Every original metadata column is kept. The join uses the row ID assigned when the metadata was loaded
(`observation_row`), never an image filename. Observations without predictions keep their metadata and have empty
score columns; the reason is in the error CSV (`observation_row` / `observation_id`).

In [ ]:
final_df = work_metadata_df.merge(results_df, on="_observation_row", how="left", validate="one_to_one")
final_df = final_df.rename(columns={"_observation_row": "observation_row"})

preferred_columns = [
    # Source, identity, and timing references
    "observation_row", OBS_ID_COL, "person_id", "frame_id", "video_id", "video_title", VIDEO_COL, FRAME_COL,
    "source_time_seconds", "source_time_second", "det_score",
    # Crop actually used
    "bbox_x1_used", "bbox_y1_used", "bbox_x2_used", "bbox_y2_used",
    # Model and suggested labels
    "model_name", "suggested_label", "positive_score", "negative_score", "neutral_score",
    "top_derived_score", "top_vs_second_margin", "review_flag",
    # Original expression-class scores
    "raw_happy_score", "raw_sad_score", "raw_angry_score", "raw_fear_score", "raw_disgust_score",
    "raw_neutral_score", "raw_surprise_score",
    # Human review fields
    "review_status", "reviewer_label", "reviewer_notes",
]
first_columns = list(dict.fromkeys(c for c in preferred_columns if c is not None and c in final_df.columns))
final_df = final_df[first_columns + [c for c in final_df.columns if c not in first_columns]]

final_df.to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")
errors_df.to_csv(ERROR_CSV, index=False, encoding="utf-8-sig")
final_df.head(50).to_csv(SAMPLE_CSV, index=False, encoding="utf-8-sig")

print("Results CSV:", FINAL_CSV.resolve())
print("Errors CSV: ", ERROR_CSV.resolve())
print("Sample CSV: ", SAMPLE_CSV.resolve(), "(first 50 rows)")
if IN_COLAB and COLAB_DRIVE_OUTPUT_DIR is not None and Path("/content/drive/MyDrive").is_dir():
    shutil.copytree(OUTPUT_DIR, COLAB_DRIVE_OUTPUT_DIR, dirs_exist_ok=True)  # the runtime disk is wiped on disconnect
    print("Backed up to Drive:", COLAB_DRIVE_OUTPUT_DIR)
print(f"Rows: {len(final_df):,} | Columns: {len(final_df.columns)} | "
      f"with predictions: {final_df['suggested_label'].notna().sum():,}")
display(final_df.head(10))

## 11 — Summary and ambiguous cases

Counts of suggested labels describe the model's expression classifications of individual frames; they are not counts
of participants' actual emotional states.

In [ ]:
scored_df = final_df[final_df["suggested_label"].notna()]

if scored_df.empty:
    print("No results to summarize.")
else:
    display(scored_df["suggested_label"].value_counts(dropna=False)
            .rename_axis("suggested_label").to_frame("number_of_observations"))
    display(scored_df[["positive_score", "negative_score", "neutral_score", "raw_surprise_score"]].describe().T)
    display(scored_df.sort_values("top_vs_second_margin")[
        [c for c in ["observation_row", OBS_ID_COL, "person_id", VIDEO_COL, FRAME_COL, "suggested_label",
                     "positive_score", "negative_score", "neutral_score", "raw_surprise_score",
                     "top_vs_second_margin", "review_flag"] if c is not None]
    ].head(5))

## 12 — Inspect examples reconstructed from the videos

Faces are re-cropped from `video + frame + bbox` for display only; nothing is saved. Each group's videos are opened
once and frames are read in order.

In [ ]:
inspection_groups = [
    ("Top Positive", scored_df.nlargest(TOP_N, "positive_score")),
    ("Top Negative", scored_df.nlargest(TOP_N, "negative_score")),
    ("Top Neutral", scored_df.nlargest(TOP_N, "neutral_score")),
    ("Most Uncertain", scored_df.nsmallest(TOP_N, "top_vs_second_margin")),
]

selected = pd.concat([df for _, df in inspection_groups]).drop_duplicates("observation_row")
faces = {}
for video_value, video_df in selected.groupby(selected[VIDEO_COL].astype(str), sort=False):
    try:
        with FrameReader(resolved_video_paths[video_value]) as reader:
            for _, row in video_df.sort_values(FRAME_COL).iterrows():
                try:
                    frame_bgr = reader.read(int(row[FRAME_COL]) + FRAME_INDEX_OFFSET)
                    faces[row["observation_row"]] = crop_face_rgb(frame_bgr, bbox_from_row(row))[0]
                except Exception as exc:
                    faces[row["observation_row"]] = exc
    except Exception as exc:
        for observation_row in video_df["observation_row"]:
            faces[observation_row] = exc

for heading, group_df in inspection_groups:
    print(f"\n{heading} — top {len(group_df)}")
    if group_df.empty:
        print("No results available.")
        continue
    fig, axes = plt.subplots(1, len(group_df), figsize=(2.6 * len(group_df), 3.2), squeeze=False)
    for ax, (_, row) in zip(axes[0], group_df.iterrows()):
        face = faces.get(row["observation_row"])
        if isinstance(face, np.ndarray):
            ax.imshow(face)
        else:
            ax.text(0.5, 0.5, str(face), ha="center", va="center", wrap=True, fontsize=6)
        ax.axis("off")
        who = row["person_id"] if "person_id" in row.index else f"row {row['observation_row']}"
        ax.set_title(f"{who} · f{int(row[FRAME_COL])}\n"
                     f"P {row['positive_score']:.2f} | N {row['negative_score']:.2f} | "
                     f"Neu {row['neutral_score']:.2f}\nmargin {row['top_vs_second_margin']:.2f}", fontsize=7)
    plt.tight_layout()
    plt.show()

## Notes

- **Speed.** Runtime is dominated by video decoding for long videos and by inference for dense, many-face videos.
  Raise `GPU_BATCH_SIZE` if GPU memory allows; lower it if a batch fails with an out-of-memory error (logged in the
  error CSV). Reading videos from local runtime disk is faster than from mounted Drive.
- **Frame indexing.** Frame numbers are treated as 0-based source-frame indices (`FRAME_INDEX_OFFSET` adjusts this).
  Use the test cell to confirm the crop shows the intended face before a full run.
- **Outputs.** `sentiment_outputs/` contains per-observation scores linked to person IDs. Keep it out of version
  control and share it only within the authorized project team.